# SocialChoice-06 (C#) : Mobius sur le treillis des coalitions — aggregation, pouvoir et manipulation

**Navigation** : [jumeau Python](ChoixSocial-06-Mobius-Aggregation-Pouvoir-Manipulation-Python.ipynb) | [SC-05](ChoixSocial-05-Gibbard-Satterthwaite-Python.ipynb) | [SC-07 >>](ChoixSocial-07-Committees-Core-Python.ipynb) | [README](README.md) | [GameTheory](../README.md) | [jumeau Rust](ChoixSocial-06-Mobius-Aggregation-Pouvoir-Manipulation-Rust.ipynb)

Ce carnet est le **jumeau C#** de SC-06 Python : même loi, même jeu, même témoin — exécutés sous le kernel `.NET (C#)`, **BCL et LINQ uniquement** (aucune dépendance NuGet). Il atteste que l'inversion de Mobius, la valeur de Shapley par dividendes et le témoin de manipulation de Gibbard-Satterthwaite se formalisent et s'exécutent de manière identique sur un second substrat.

La loi centrale — celle que ce notebook atteste — est l'**inversion de Mobius sur le treillis des coalitions** :

$$v(S) \;=\; \sum_{T \subseteq S} m(T), \qquad m(T) \;=\; \sum_{R \subseteq T} (-1)^{|T|-|R|}\, v(R)$$

Le coefficient $m(T)$ est le **dividende de Harsanyi** : la valeur *pure* que la coalition $T$ apporte au-delà de ce que ses sous-coalitions produisent déjà seules. Le **témoin** reste celui de Gibbard-Satterthwaite : une stratégie de manipulation profitable, mesurée ici sur un électorat **pondéré**.

## 1. Le modèle : jeux de vote pondérés et dividende de Harsanyi

Un **jeu de vote pondéré** $[q; w_1, \dots, w_n]$ définit $v(S) = 1$ si $\sum_{i \in S} w_i \geq q$ (la coalition gagne), $v(S) = 0$ sinon. C'est le modèle du conseil, de l'assemblée, du Conseil européen — et le pont exact entre le vote ordinal de SC-03/SC-05 et la théorie coopérative.

Sur ce jeu simple (à valeurs dans $\{0, 1\}$), l'inversion de Mobius reste définie : $m(T)$ mesure la *synergie pure* de $T$. Une coalition gagnante dont tous les sous-ensembles propres perdent porte $m(T) = 1$ ; les grandes coalitions peuvent porter des dividendes **négatifs**. Nous travaillerons sur l'exemple canonique $[6; 4, 3, 2, 1]$ (4 joueurs, quota 6).

### Implémentation : inversion de Mobius et reconstruction

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

const int N_PLAYERS = 4;

// Organe partage du carnet : les fonctions aides et le jeu [6; 4, 3, 2, 1].
// Declarees en classe (et non en fonctions locales) car les fonctions locales
// d'une cellule .NET Interactive ne persistent pas aux cellules suivantes.
class Mobius
{
    public static int[] Weights = { 4, 3, 2, 1 };   // joueurs 1..4
    public static int Quota = 6;

    // Toutes les coalitions de {0..n-1}, par taille croissante.
    public static IEnumerable<List<int>> AllCoalitions(int n)
    {
        for (int size = 0; size <= n; size++)
            foreach (var combo in Combinations(Enumerable.Range(0, n), size))
                yield return combo;
    }

    // k-combinaisons d'une source (ordre croissant) — BCL/LINQ uniquement.
    public static IEnumerable<List<int>> Combinations(IEnumerable<int> source, int k)
    {
        var result = new List<List<int>>();
        Combine(source.ToList(), 0, k, new List<int>(), result);
        return result;
    }

    static void Combine(List<int> items, int start, int k, List<int> current, List<List<int>> result)
    {
        if (k == 0) { result.Add(new List<int>(current)); return; }
        for (int i = start; i <= items.Count - k; i++)
        {
            current.Add(items[i]);
            Combine(items, i + 1, k - 1, current, result);
            current.RemoveAt(current.Count - 1);
        }
    }

    // Jeu de vote pondere [6; 4, 3, 2, 1] : 1 si la coalition atteint le quota.
    public static int V(IEnumerable<int> S) => S.Sum(i => Weights[i]) >= Quota ? 1 : 0;

    // Dividende de Harsanyi m(T) = somme_{R subset T} (-1)^(|T|-|R|) v(R).
    public static int MobiusCoeff(IEnumerable<int> S)
    {
        var s = S.ToList();
        int total = 0;
        for (int size = 0; size <= s.Count; size++)
        {
            int sign = ((s.Count - size) % 2 == 0) ? 1 : -1;
            foreach (var R in Combinations(s, size))
                total += sign * V(R);
        }
        return total;
    }

    // Reconstruction v(S) = somme_{T subseteq S} m(T).
    public static int MobiusReconstruct(IEnumerable<int> S)
    {
        var s = S.ToList();
        int total = 0;
        for (int size = 0; size <= s.Count; size++)
            foreach (var T in Combinations(s, size))
                total += MobiusCoeff(T);
        return total;
    }
}

// --- Verification de la loi : reconstruction exacte sur TOUTES les coalitions ---
bool roundtripOk = Mobius.AllCoalitions(N_PLAYERS)
    .All(S => Mobius.MobiusReconstruct(S) == Mobius.V(S));
Console.WriteLine($"Loi de reconstruction (v = somme des dividendes) verifiee sur les 16 coalitions : {roundtripOk}");

// --- Table des dividendes non nuls ---
Console.WriteLine();
Console.WriteLine("Dividendes de Harsanyi non nuls du jeu [6; 4, 3, 2, 1] :");
Console.WriteLine(string.Format("{0,-12}{1,-7}{2,-5}{3,-5}", "coalition", "poids", "v(S)", "m(T)"));
foreach (var S in Mobius.AllCoalitions(N_PLAYERS))
{
    int m = Mobius.MobiusCoeff(S);
    if (m != 0)
    {
        int poids = S.Count > 0 ? S.Sum(i => Mobius.Weights[i]) : 0;
        string nom = "{" + string.Join(",", S.Select(i => i + 1)) + "}";
        Console.WriteLine($"{nom,-12}{poids,-7}{Mobius.V(S),-5}{m,-5}");
    }
}

Loi de reconstruction (v = somme des dividendes) verifiee sur les 16 coalitions : True


Dividendes de Harsanyi non nuls du jeu [6; 4, 3, 2, 1] :


coalition   poids  v(S) m(T) 


{1,2}       7      1    1    


{1,3}       6      1    1    


{1,2,3}     9      1    -1   


{2,3,4}     6      1    1    


{1,2,3,4}   10     1    -1   


### Lecture du résultat

La reconstruction est **exacte** sur les 16 coalitions : la loi $v = \sum m(T) \cdot u_T$ tient, même sur un jeu à valeurs binaires. La table des dividendes dit quelque chose que la table des poids ne dit pas :

- les trois coalitions **minimalement gagnantes** — {1,3} (poids 6), {1,2} (poids 7) et {2,3,4} (poids 6) — portent chacune un dividende de $+1$ : c'est là que vit la synergie pure ;
- la coalition {1,2,3} porte un dividende de $-1$ : elle ajoute moins que la somme de ses parties, car ses sous-coalitions {1,2} et {1,3} gagnent déjà ;
- la grande coalition porte aussi $-1$, pour la même raison.

C'est la promesse de la loi de Mobius : elle rend la structure de pouvoir **lisible coefficient par coefficient**, là où la fonction $v$ brute ne montre qu'un mur de 0 et de 1.

## 2. Le pouvoir via les dividendes : valeur de Shapley et indice de Banzhaf

Le pont vers la théorie coopérative passe par une identité remarquable : la **valeur de Shapley** d'un joueur est la somme de ses dividendes de Harsanyi, **partagés à parts égales** au sein de chaque coalition,

$$\phi_i(v) \;=\; \sum_{T \ni i} \frac{m(T)}{|T|},$$

L'indice de **Banzhaf**, lui, compte les balanciers (swings) : les coalitions $S \ni i$ gagnantes devenues perdantes quand $i$ les quitte. Nous calculons la valeur de Shapley **deux fois, par des voies indépendantes** — via les dividendes (la loi de ce notebook), puis par énumération directe des $n!$ ordres d'entrée (la définition classique) — puis nous confrontons à l'indice de Banzhaf.

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

// (N_PLAYERS, la classe Mobius et le jeu sont persistants depuis la cellule 3)

// Valeur de Shapley via les dividendes de Harsanyi, partagees a parts egales.
double ShapleyViaDividends(int i, int n)
{
    double total = 0.0;
    for (int size = 1; size <= n; size++)
        foreach (var T in Mobius.Combinations(Enumerable.Range(0, n), size))
            if (T.Contains(i))
                total += (double)Mobius.MobiusCoeff(T) / size;   // cast double : l'entier/int entier tronquerait
    return total;
}

// Definition classique : contribution marginale moyenne sur les n! ordres d'entree.
double ShapleyDirect(int i, int n)
{
    double total = 0;
    foreach (var ordre in Permutations(Enumerable.Range(0, n)))
    {
        int k = ordre.IndexOf(i);
        var avant = ordre.Take(k).ToList();
        var avec = new List<int>(avant) { i };
        total += Mobius.V(avec) - Mobius.V(avant);
    }
    return total / Factorial(n);
}

static IEnumerable<List<int>> Permutations(IEnumerable<int> source)
{
    var result = new List<List<int>>();
    Perm(source.ToList(), 0, result);
    return result;
}

static void Perm(List<int> items, int start, List<List<int>> result)
{
    if (start == items.Count) { result.Add(new List<int>(items)); return; }
    for (int i = start; i < items.Count; i++)
    {
        (items[start], items[i]) = (items[i], items[start]);
        Perm(items, start + 1, result);
        (items[start], items[i]) = (items[i], items[start]);
    }
}

static double Factorial(int n) { double r = 1; for (int i = 2; i <= n; i++) r *= i; return r; }

// Nombre de balanciers du joueur i : coalitions gagnantes S avec S prive de i perdante.
int BanzhafSwings(int i, int n)
{
    int count = 0;
    for (int size = 1; size <= n; size++)
        foreach (var S in Mobius.Combinations(Enumerable.Range(0, n), size))
            if (S.Contains(i))
            {
                var sansI = S.Where(p => p != i).ToList();
                if (Mobius.V(S) == 1 && Mobius.V(sansI) == 0)
                    count++;
            }
    return count;
}

int[] swings = Enumerable.Range(0, N_PLAYERS).Select(i => BanzhafSwings(i, N_PLAYERS)).ToArray();
int totalSwings = swings.Sum();
Console.WriteLine(string.Format("{0,-8}{1,-7}{2,-12}{3,-16}{4,-18}{5,-15}",
    "joueur", "poids", "part poids", "Shapley (m(T))", "Shapley (ordres)", "Banzhaf norm."));
for (int i = 0; i < N_PLAYERS; i++)
{
    double sh = ShapleyViaDividends(i, N_PLAYERS);
    double sd = ShapleyDirect(i, N_PLAYERS);
    double bz = (double)swings[i] / totalSwings;
    double part = (double)Mobius.Weights[i] / Mobius.Weights.Sum();
    Console.WriteLine(string.Format("{0,-8}{1,-7}{2,-12:P2}{3,-16:F4}{4,-18:F4}{5,-15:F4}",
        i + 1, Mobius.Weights[i], part, sh, sd, bz));
}

// Double controle de coherence
double phiTotal = Enumerable.Range(0, N_PLAYERS).Sum(i => ShapleyViaDividends(i, N_PLAYERS));
double ecartDivDirect = Enumerable.Range(0, N_PLAYERS)
    .Max(i => Math.Abs(ShapleyViaDividends(i, N_PLAYERS) - ShapleyDirect(i, N_PLAYERS)));
int vN = Mobius.V(Enumerable.Range(0, N_PLAYERS));
Console.WriteLine();
Console.WriteLine(string.Format("Somme des valeurs de Shapley = {0:0.######} | v(N) = {1}", phiTotal, vN));
Console.WriteLine(string.Format("Ecart max |Shapley via dividendes - Shapley par ordres| = {0:0.##e}", ecartDivDirect));
if (Math.Abs(phiTotal - vN) >= 1e-9)
    throw new Exception("Shapley doit etre efficiente");
if (ecartDivDirect >= 1e-12)
    throw new Exception("les deux computations doivent coincider exactement");
Console.WriteLine("Efficiency de Shapley et accord des deux voies : verifies.");

joueur  poids  part poids  Shapley (m(T))  Shapley (ordres)  Banzhaf norm.  


1       4      40,00 %     0,4167          0,4167            0,4167         


2       3      30,00 %     0,2500          0,2500            0,2500         


3       2      20,00 %     0,2500          0,2500            0,2500         


4       1      10,00 %     0,0833          0,0833            0,0833         


Somme des valeurs de Shapley = 1 | v(N) = 1


Ecart max |Shapley via dividendes - Shapley par ordres| = 0e


Efficiency de Shapley et accord des deux voies : verifies.


### Lecture du resultat

Trois lectures, trois pieges evites par la mesure :

- **poids ne veut pas dire pouvoir, dans les deux sens** : le joueur 2 (poids 3, 30 % du total) et
  le joueur 3 (poids 2, 20 %) ont **exactement le meme pouvoir** (0,25 chacun) car ils sont
  interchangeables dans la structure des coalitions gagnantes ; le joueur 4 (10 % du poids) n'a
  que 8,3 % du pouvoir, mais il n'est **pas un dummy** : il est balancier dans {2,3,4} (poids 6
  avec lui, 5 sans lui). Un oeil sur la seule table des poids aurait declare les rapports
  4:3:2:1 ; le pouvoir mesure est 5:3:3:1.
- **la loi de Mobius porte bien le calcul** : la valeur calculee via les dividendes et celle
  calculee par enumeration des 24 ordres d'entree coincident exactement - deux voies
  independantes, un seul nombre. C'est le contenu du pont Lean `phi_weightedUnanimity` +
  `shapley_uniqueness` (Shapley.lean), execute ici au lieu d'etre prouve.
- **une coincidence remarquable, a ne pas generaliser** : sur ce jeu precis, Shapley et Banzhaf
  (normalise) rendent la **meme distribution** (5/12, 1/4, 1/4, 1/12). Ce n'est pas une loi :
  Shapley pondere chaque balancier par le nombre d'ordres d'entree qui l'activent, Banzhaf les
  compte tous egaux - sur un autre jeu les deux divergent (l'exercice du Conseil 1958 plus bas
  en donnera l'occasion).

C'est exactement la structure que le lac Lean atteste formellement (Shapley.lean, sections
`ShapleyValue` puis `Mobius`) : meme loi, substrat different - ici mesuree, la-bas prouvee.

## 3. Exercice 1 : la loi sur un second jeu

Verifier une loi sur un seul exemple ne prouve rien - l'exercice est de la **re-tester** sur le jeu
pondere $[8; 5, 4, 3, 2, 1]$ (5 joueurs, quota 8) : reconstruction exacte sur les $2^5 = 32$
coalitions, puis lecture de la table des dividendes.

**Attendu** : le round-trip tient sur les 32 coalitions ; les dividendes positifs majeurs vivent
sur les coalitions minimalement gagnantes ({1,2}, {1,3}, {1,4,5}, {2,3,4}, {2,3,5}) ; et la grande
coalition porte un dividende de $+2$ - a confronter au $-1$ du jeu a 4 joueurs.

In [3]:
// --- Exercice 1 : re-verifier la loi de reconstruction sur [8; 5, 4, 3, 2, 1] ---
// TODO etudiant :
//   1. redefinir les poids { 5, 4, 3, 2, 1 } et le quota 8 (nouvelle instance de jeu)
//   2. definir v_ex(S) sur ce jeu, puis reutiliser le schema de la section 1
//      (MobiusCoeff / MobiusReconstruct adaptes a v_ex)
//   3. verifier le round-trip sur les 32 coalitions et lister les dividendes non nuls
// Indice : factoriser v_ex en parametre des fonctions evite de dupliquer le code.
// Attendu : roundtrip_ok_ex = True ; dividendes positifs sur les 5 minimalement gagnantes ;
// m(grande coalition) = +2.

Console.WriteLine("Exercice 1 a completer : loi de Mobius sur [8; 5, 4, 3, 2, 1]");
List<int> resultEx1 = null;  // TODO etudiant

Exercice 1 a completer : loi de Mobius sur [8; 5, 4, 3, 2, 1]


### Question 1

Pourquoi la grande coalition porte-t-elle un dividende de $+2$ sur $[8; 5, 4, 3, 2, 1]$ alors
qu'elle portait $-1$ sur $[6; 4, 3, 2, 1]$ ? (Compter, par taille, les coalitions gagnantes de
chacun des deux jeux.) Que dit ce signe sur la synergie pure de la cooperation complete dans chaque
cas ?

In [4]:
// --- Question 1 : decomposer m(N) par taille de coalition ---
// TODO etudiant : pour chaque jeu, compter les coalitions gagnantes par taille (2, 3, 4, 5)
// et verifier que m(N) = somme des (-1)^(n-|S|) * nb_gagnantes(|S|).
// Attendu : [6;4,3,2,1] -> 2 gagnantes de taille 2, 4 de taille 3, 1 de taille 4 : m(N) = -1 ;
// [8;5,4,3,2,1] -> 2 de taille 2, 8 de taille 3, 5 de taille 4, 1 de taille 5 : m(N) = +2.

Console.WriteLine("Exercice (question 1) a completer : decomposer m(N) par taille");
List<int> resultQ1 = null;  // TODO etudiant

Exercice (question 1) a completer : decomposer m(N) par taille


## 4. Exercice 2 : chasser un dummy - le Luxembourg du Conseil de 1958

Un joueur est un **dummy** quand il n'est balancier dans aucune coalition (indice de Banzhaf nul).
Le jeu de vote du Conseil europeen de la communaute economique europeenne, 1958 (traite de Rome) :
France, Allemagne, Italie 4 voix chacune, Belgique et Pays-Bas 2, Luxembourg 1 - quota 12 :
$[12; 4, 4, 4, 2, 2, 1]$.

**Attendu** : le Luxembourg (joueur 6) est un dummy - un poids de vote reel, un pouvoir de zero -
tandis que la Belgique (joueur 4) ne l'est pas (balancier dans {F, D, B, N} : 12 avec elle, 10 sans
elle). C'est l'exemple historique qui a fonde la lecture « poids ne veut pas dire pouvoir ».

In [5]:
// --- Exercice 2 : identifier le dummy du Conseil 1958 [12; 4, 4, 4, 2, 2, 1] ---
// TODO etudiant :
//   1. definir v_tq(S) pour ce jeu (poids [4, 4, 4, 2, 2, 1], quota 12)
//   2. adapter BanzhafSwings et lister les balanciers de chaque joueur
//   3. identifier le(s) joueur(s) a zero balancier
// Indice : un balancier pour le Luxembourg exige une coalition de poids exactement 12 le contenant,
// donc 11 pour les cinq autres - or {4, 4, 4, 2, 2} ne produit que des sommes paires.
// Attendu : Luxembourg = 0 balancier (dummy) ; Belgique >= 1 balancier.

Console.WriteLine("Exercice 2 a completer : dummy du Conseil 1958");
List<int> resultEx2 = null;  // TODO etudiant

Exercice 2 a completer : dummy du Conseil 1958


## 5. Exercice 3 : le temoin - manipulation d'un electorat pondere

Le temoin exige par la loi de ce notebook n'est pas une courbe : c'est une **strategie de
manipulation profitable** (Gibbard-Satterthwaite). Cadre : les 4 joueurs du jeu $[6; 4, 3, 2, 1]$
votent sur 3 alternatives $A, B, C$ au **Borda pondere** - chaque bulletin donne 2, 1, 0 points a
ses alternatives rangees, multiplies par le poids du votant.

Profil sincere :

| Votant | Poids | Classement sincere |
|---|---|---|
| V1 | 4 | $B > A > C$ |
| V2 | 3 | $A > C > B$ |
| V3 | 2 | $C > B > A$ |
| V4 | 1 | $C > A > B$ |

La question : **V3 (poids 2, sincere $C > B > A$) peut-il obtenir un resultat strictement meilleur
en votant autrement ?** Enumerer ses $3! = 6$ bulletins possibles, calculer le gagnant Borda
pondere pour chacun, et ne garder que les bulletins insinceres **strictement profitables au sens
du classement sincere** de V3 - c'est-a-dire faisant elire une alternative qu'il prefere au gagnant
sincere.

**Attendu** : au moins un bulletin insincere strictement profitable - le temoin constructif de
Gibbard-Satterthwaite, cette fois sur substrat pondere : manipuler, c'est deplacer son poids a
travers les coalitions que la decomposition de Mobius rend visibles.

In [6]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] ALTS = { "A", "B", "C" };

// Profil sincere : votant -> (poids, classement).
var profilsSinceres = new Dictionary<string, (int poids, List<string> classement)>
{
    { "V1", (4, new List<string> { "B", "A", "C" }) },
    { "V2", (3, new List<string> { "A", "C", "B" }) },
    { "V3", (2, new List<string> { "C", "B", "A" }) },
    { "V4", (1, new List<string> { "C", "A", "B" }) },
};

// Gagnant au Borda pondere : chaque bulletin donne (n-1-position) points, multiplie par le poids.
(string gagnant, Dictionary<string, int> scores) BordaPondereGagnant(
    Dictionary<string, (int poids, List<string> classement)> bulletins)
{
    var scores = new Dictionary<string, int>();
    foreach (var a in ALTS) scores[a] = 0;
    foreach (var (poids, classement) in bulletins.Values)
        for (int position = 0; position < classement.Count; position++)
            scores[classement[position]] += poids * (ALTS.Length - 1 - position);
    // Gagnant = score maximal ; en cas d'ex-aequo, l'alternative alphabetiquement premiere l'emporte.
    var gagnant = scores.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key).First().Key;
    return (gagnant, scores);
}

var (gagnantSincere, scoresSinceres) = BordaPondereGagnant(profilsSinceres);
Console.WriteLine("Scores Borda ponderes sous vote sincere : " +
    string.Join(", ", ALTS.Select(a => a + "=" + scoresSinceres[a])));
Console.WriteLine("Gagnant sincere : " + gagnantSincere);

// --- Exercice 3 : enumerer les 6 bulletins de V3 et trouver une manipulation profitable ---
// TODO etudiant :
//   1. pour chaque classement possible de V3 (6 permutations de A, B, C),
//      construire le profil modifie et calculer le gagnant Borda pondere
//   2. ne garder que les bulletins ININCERES strictement profitables au sens du
//      classement sincere de V3 (C > B > A) : le gagnant manipule doit etre mieux
//      classe que le gagnant sincere dans ["C", "B", "A"]
//   3. exhiber le bulletin manipulateur et le couple (gagnant sincere, gagnant manipule)
// Indice : comparer les positions des deux gagnants dans ["C", "B", "A"] - l'indice
// le plus petit gagne au sens de V3.
// Attendu : au moins une manipulation profitable ; l'annoter explicitement.

Console.WriteLine("Exercice 3 a completer : temoin de manipulation de V3 sous Borda pondere");
List<int> resultEx3 = null;  // TODO etudiant

Scores Borda ponderes sous vote sincere : A=11, B=10, C=9


Gagnant sincere : A


Exercice 3 a completer : temoin de manipulation de V3 sous Borda pondere


### Lecture attendue du temoin

Ce que le temoin doit montrer une fois complete :

1. le vote sincere produit un gagnant **mesure** (scores calcules, pas intuition) ;
2. il existe un bulletin insincere de V3 qui fait elire une alternative que V3 prefere **au sens de
son classement sincere** - la definition exacte de la manipulabilite ;
3. le mecanisme du gain se lit dans la structure ponderee : le poids 2 de V3 bascule les coalitions
gagnantes d'une alternative vers une autre. La decomposition de Mobius de la section 1 etait la
carte ; la manipulation est le deplacement du poids sur cette carte.

Noter l'asymetrie que le cadre pondere ajoute a SC-05 : V1 (poids 4) et V2 (poids 3) n'ont pas
les memes capacites de manipulation que V3 (poids 2) - la strategie profitable n'est pas
distribuee uniformement, elle epouse la geometrie du pouvoir. Gibbard-Satterthwaite garantit
qu'aucune regle raisonnable n'echappe au temoin ; il ne dit pas que tous les votants y ont le meme
acces.

## 6. Conclusion : ce que la loi et le temoin attestent

**Trois resultats a retenir** :

1. **La loi tient, mesuree** : la reconstruction $v = \sum_T m(T) \cdot u_T$ est exacte sur
chaque coalition testee - l'agregation d'un collectif cooperatif se decompose de facon unique en
dividendes de Harsanyi. Le lac Lean du depot
([`Shapley.lean`](../game_theory_lean/CooperativeGames/Shapley.lean)) prouve cette meme loi
formellement (`Mobius.mobiusCoeff`, `Mobius.mobiusReconstruction`, sorry-free) ; ce notebook en est
le **temoin C# execute** - meme loi, deux substrats independants
([EPIC #12204](https://github.com/jsboige/CoursIA/issues/12204), operation 14 « Agreger un
collectif »).

2. **Le pouvoir ne se lit pas dans les poids** : sur $[6; 4, 3, 2, 1]$, joueurs 2 et 3 ont le meme
pouvoir malgre des poids de 3 et 2 ; sur le Conseil de 1958, le Luxembourg pesait 1 voix pour un
pouvoir nul - un dummy a poids non nul, l'illustration historique de l'ecart poids/pouvoir.

3. **Le temoin est constructif, y compris sous pondération** : une strategie de vote insincere
strictement profitable existe et s'exhibe par enumeration - Gibbard-Satterthwaite instancie sur
un electorat pondere, ou manipuler revient a deplacer son poids a travers le treillis des
coalitions.

**Prolongements** : l'indice de pouvoir des membres du Conseil europeen actuel (a double majorite),
les jeux composes ou des dummies redeviennent actifs, et la formalisation Lean complete dans le
notebook [SC-02](ChoixSocial-01b-Lean-SocialChoice-Formal-Lean.ipynb).